# 第 5 章 客户分群与差异化运营

本章商业问题：**用户不是同一种人，如何分成能指导运营动作的群体？**

本章所有代码默认优先读取真实 ETL 接口 `http://192.168.31.47:38173/api/etl`。如果接口暂时不可用，会自动回退到本地 SQLite 后备数据，保证课堂可以继续进行。

## 0. 先建立商业问题意识

在商业课里，代码不是第一步。第一步是判断：这个问题影响收入、成本、用户体验、库存风险，还是营销效率？只有先说清楚商业目标，后面的数据选择和模型选择才不会跑偏。

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd()
if COURSE_ROOT.name in ["notebooks", "student_notebooks", "teacher_notebooks"]:
    COURSE_ROOT = COURSE_ROOT.parent
elif not (COURSE_ROOT / "course_utils").exists():
    COURSE_ROOT = Path("..").resolve()

sys.path.insert(0, str(COURSE_ROOT))

from course_utils.data_loader import (
    API_BASE, load_table, get_metrics, get_quality_report,
    get_table_catalog, get_schema, paid_orders, api_status, query_table
)
from course_utils.business import money, pct, section

try:
    import matplotlib.pyplot as plt
    plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS", "DejaVu Sans"]
    plt.rcParams["axes.unicode_minus"] = False
except Exception:
    pass

print("课程目录:", COURSE_ROOT)
print("ETL API:", API_BASE)
print("API 状态:", api_status())

## 1. 查看 ETL 数据资产

下面先查看 ETL 接口暴露了哪些表。请注意 `dim_` 开头的是维度表，通常描述对象；`fact_` 开头的是事实表，通常记录业务事件。

In [ ]:
catalog = get_table_catalog()
tables = catalog["tables"]
print("可用表数量:", catalog.get("total", len(tables)))
for t in tables[:12]:
    print(t["tableName"], t.get("recordCount"), t.get("type"), t.get("description", ""))

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
orders = paid_orders()
customer = orders.groupby("user_id").agg(order_count=("order_id", "nunique"), total_paid=("paid_amount", "sum"), avg_paid=("paid_amount", "mean")).reset_index()
X = customer[["order_count", "total_paid", "avg_paid"]]
Xs = StandardScaler().fit_transform(X)
scores = []
for k in range(2, 7):
    labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(Xs)
    scores.append((k, silhouette_score(Xs, labels)))
scores

In [ ]:
k = max(scores, key=lambda x: x[1])[0]
customer["cluster"] = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(Xs)
profile = customer.groupby("cluster").agg(users=("user_id", "count"), avg_orders=("order_count", "mean"), avg_total_paid=("total_paid", "mean"), avg_order_value=("avg_paid", "mean")).reset_index()
profile

## 商业解释

分群之后必须命名。命名不是装饰，而是把分析结果变成运营动作。例如高消费高频客适合会员权益，低频高客单客适合新品唤醒，低消费沉睡客不适合高成本触达。

In [ ]:
assert profile["users"].sum() == len(customer)
print("第 05 章验证通过")

## 学生作业

请补充 150 到 300 字商业解读，至少引用两个数据结果，并给出一个明确运营动作。